# V4.7










### 分析中notebook

|名称|notebook名称|notebook URL|
|----|----|----|
|社長と従業員の行動範囲|range_of_action|**https://www.kaggle.com/code/nagatakengo/range-of-action**<br><br><br>|
|コストと物価|costs|**https://www.kaggle.com/code/nagatakengo/costs**<br><br><br>|
|未来のメロンの価格を予測する|predict-future-melon-prices|**https://www.kaggle.com/code/nagatakengo/predict-future-melon-prices**<br><br><br>|

# 提出関数

In [1]:
%%writefile main.py




hire_control_states = {}


def get_market_price(obs, product):
    """現在価格を取得する。"""
    return obs["market"]["prices"][product]


def score(input):
    """max_depth=3 のDecisionTreeモデル。"""

    if input[3] <= 10096.5:
        if input[0] <= 257.5:
            if input[2] <= 265.0:
                var0 = 262.30210213139543
            else:
                var0 = 269.14311998948614
        else:
            if input[2] <= 180.0:
                var0 = 169.52286374133948
            else:
                var0 = 192.71375464684016
    else:
        if input[0] <= 497.5:
            if input[3] <= 10127.5:
                var0 = 95.84500378501136
            else:
                var0 = 82.26066931619805
        else:
            if input[3] <= 10154.5:
                var0 = 19.437829958238122
            else:
                var0 = 6.73007806147341

    return var0


def should_sell_melon(
    melon_price,
    melon_stock,
    melon_in_shed,
    day,
    step,
):
    """MELONをSELLするかHOLDするか判断する。"""

    # 終盤は価格に関係なく売却
    if step >= 710:
        return True

    # 12ターン以内の最高価格を予測
    pred_price = score([
        step,
        melon_in_shed,
        melon_price,
        melon_stock,
        0,
        0,
        0,
    ])

    # 現在価格の方が予測最高価格以上なら売却
    if melon_price >= pred_price:
        return True

    return False

def should_sell_milk(milk_price, step,):
    """MILKをSELLするかHOLDするか判断する"""

    if step >= 710:
        return True

    if milk_price >= 160:
        return True

    return False


def get_harvest_age(crop_name):
    """現在の戦略で使う収穫開始日を返す。"""
    if crop_name == "MELON":
        return 10

    if crop_name == "STRAWBERRY":
        return 10

    return 2


def get_tile_action(tile, day, allow_fertilizer_collection=True,):
    """足元のタイルで今すぐ行う作業を返す。"""
    if not isinstance(tile, dict):
        return None

    if tile.get("kind") == "WEED":
        return ["DIG"]

    if tile.get("animal"):
        if not tile.get("fed_today", False):
            return ["FEED"]

        if not tile.get("cared_today", False):
            return ["CARE"]

        if tile.get("yield_units", 0) > 0:
            return ["HARVEST"]

        if (
            allow_fertilizer_collection
            and tile.get("fertilizer_available", 0) > 0
        ):
            return ["COLLECT_FERTILIZER"]

        return None

    if tile.get("kind") != "PLANT":
        return None

    crop_name = tile.get("crop", "WHEAT")
    crop_age = day - tile.get("planted_day", day)
    harvest_age = get_harvest_age(crop_name)

    if not tile.get("watered_today", True):
        return ["WATER"]

    if crop_name == "STRAWBERRY":
        if (crop_age >= harvest_age and tile.get("yield_units", 0) > 0):
            return ["HARVEST"]

        return None

    if crop_age >= harvest_age:
        return ["HARVEST"]

    return None


def step_toward(fx, fy, tx, ty, tiles):
    """
    目的地に近づく方向へ1マス移動する。
    目的地へ直接進めない場合は、移動可能な方向を選ぶ。
    """
    max_y = len(tiles) - 1
    max_x = len(tiles[0]) - 1

    if fx == tx and fy == ty:
        return "PASS"

    candidates = []

    if fx < tx and fx < max_x and tiles[fy][fx + 1] != "LOCKED":
        candidates.append("EAST")
    elif fx > tx and fx > 0 and tiles[fy][fx - 1] != "LOCKED":
        candidates.append("WEST")

    if fy < ty and fy < max_y and tiles[fy + 1][fx] != "LOCKED":
        candidates.append("SOUTH")
    elif fy > ty and fy > 0 and tiles[fy - 1][fx] != "LOCKED":
        candidates.append("NORTH")

    if candidates:
        x_distance = abs(tx - fx)
        y_distance = abs(ty - fy)

        if x_distance >= y_distance:
            if "EAST" in candidates:
                return "EAST"

            if "WEST" in candidates:
                return "WEST"

        if "SOUTH" in candidates:
            return "SOUTH"

        if "NORTH" in candidates:
            return "NORTH"

        return candidates[0]

    valid_dirs = []

    if fy > 0 and tiles[fy - 1][fx] != "LOCKED":
        valid_dirs.append("NORTH")

    if fy < max_y and tiles[fy + 1][fx] != "LOCKED":
        valid_dirs.append("SOUTH")

    if fx < max_x and tiles[fy][fx + 1] != "LOCKED":
        valid_dirs.append("EAST")

    if fx > 0 and tiles[fy][fx - 1] != "LOCKED":
        valid_dirs.append("WEST")

    if valid_dirs:
        direction_offsets = {
            "NORTH": (0, -1),
            "SOUTH": (0, 1),
            "EAST": (1, 0),
            "WEST": (-1, 0),
        }

        best_dir = None
        best_distance = 9999

        for move_dir in valid_dirs:
            dx, dy = direction_offsets[move_dir]

            next_x = fx + dx
            next_y = fy + dy

            distance = (abs(tx - next_x) + abs(ty - next_y))

            if distance < best_distance:
                best_distance = distance
                best_dir = move_dir

        return best_dir

    return "PASS"


def find_empty_pasture(
    tiles,
    fx,
    fy,
):
    """現在位置から最も近い空のpastureを返す"""
    best_target = None
    best_distance = 9999

    for y in range(len(tiles)):
        for x in range(len(tiles[0])):

            tile = tiles[y][x]

            if (
                isinstance(tile, dict)
                and tile.get("kind") == "PASTURE"
                and not tile.get("animal")
            ):
                distance = (abs(x - fx) + abs(y - fy))

                if distance < best_distance:
                    best_distance = distance
                    best_target = (x, y)

    return best_target


def find_fertilize_target(
    tiles,
    fx,
    fy,
    day,
):
    """現在位置から最も近い肥料対象の植物を返す"""

    best_target = None
    best_distance = 9999

    for y in range(len(tiles)):
        for x in range(len(tiles[0])):

            tile = tiles[y][x]

            if(
                isinstance(tile, dict)
                and tile.get("kind") == "PLANT"
                and tile.get("fertilized_until_day", -1) < day
            ):
                distance = abs(x - fx) + abs(y - fy)

                if distance < best_distance:
                    best_distance = distance
                    best_target = (x, y)

    return best_target


def find_cow_target(
    tiles,
    fx,
    fy,
    day,
    allow_fertilizer_collection=True,
):
    """現在位置から最も近い作業対象のCOWを返す"""

    best_target = None
    best_distance = 9999

    for y in range(len(tiles)):
        for x in range(len(tiles[0])):

            tile = tiles[y][x]

            if(
                isinstance(tile, dict)
                and tile.get("animal") == "COW"
                and get_tile_action(
                    tile,
                    day,
                    allow_fertilizer_collection,
                ) is not None
            ):
                distance = abs(x - fx) + abs(y - fy)

                if distance < best_distance:
                    best_distance = distance
                    best_target = (x, y)

    return best_target

def get_hand_area(hand_index):
    """作業員ごとの担当エリアを返す"""

    hand_areas = {
        0: "NW",
        1: "NE",
        2: "SW",
        3: "NE",
        4: "SW",
        5: "NW",
        6: "NE",
        7: "NW",
    }

    return hand_areas.get(hand_index)


def find_target_tile(
    tiles,
    fx,
    fy,
    has_seeds,
    day,
    excluded_coords=None,
    target_area=None,
):
    """
    現在位置から各作業候補を評価し、
    最もスコアの高いタイルの座標を返す。
    """
    if excluded_coords is None:
        excluded_coords = set()

    best_target = None
    best_score = -9999

    for y in range(len(tiles)):
        for x in range(len(tiles[0])):
            if (x, y) in excluded_coords:
                continue

            outside_area = False

            if target_area == "NW":
                if not (x < 5 and y < 5):
                    outside_area = True

            elif target_area == "NE":
                if not (x >= 5 and y < 5):
                    outside_area = True

            elif target_area == "SW":
                if not (x < 5 and y >= 5):
                    outside_area = True

            elif target_area == "SE":
                if not (x >= 5 and y >= 5):
                    outside_area = True


            tile = tiles[y][x]
            if tile == "LOCKED":
                continue

            base_score = None

            # 雑草
            if isinstance(tile, dict) and tile.get("kind") == "WEED":
                if day >= 27:
                    continue

                base_score = 0

            #動物
            elif isinstance(tile, dict) and tile.get("animal"):
                if not tile.get("fed_today", False):
                    base_score = 100

                elif not tile.get("cared_today", False):
                    base_score = 75

                elif tile.get("yield_units", 0) > 0:
                    base_score = 60

                elif tile.get("fertilizer_available", 0) > 0:
                    base_score = 55

            # 植物
            elif isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop_name = tile.get("crop", "WHEAT")
                crop_age = day - tile.get("planted_day", day)
                harvest_age = get_harvest_age(crop_name)

                if crop_name == "STRAWBERRY":
                    if(
                        not tile.get("watered_today", True)
                        and tile.get("consecutive_unwatered", 0) >= 1
                    ):
                        base_score = 75

                    elif (crop_age >= harvest_age and tile.get("yield_units", 0) > 0):
                        base_score = 25

                    elif not tile.get("watered_today", True):
                        base_score = 50

                elif crop_age >= harvest_age:
                    if crop_name == "MELON":
                        base_score = 250

                    else:
                        base_score = 25

                elif not tile.get("watered_today", True):
                    if tile.get("consecutive_unwatered", 0) >= 1:
                        base_score = 75

                    else:
                        base_score = 50

            # 空き地
            elif tile is None and has_seeds:
                if day >= 27:
                    continue

                base_score = 50

            if base_score is None:
                continue

            movement_cost = abs(x - fx) + abs(y - fy)
            task_score = base_score - movement_cost

            if outside_area:
                task_score -= 30

            if task_score > best_score:
                best_score = task_score
                best_target = (x, y)

    return best_target

def update_hire_control(
    player,
    day,
    step,
    unlocked_quads,
):
    """前日の作業員PASS率から目標作業員数を更新する。"""

    if (
        step == 0
        or player not in hire_control_states
    ):
        hire_control_states[player] = {
            "day": day,
            "pass_count": 0,
            "action_count": 0,
            "target_hands": 6,
        }

        return hire_control_states[player]

    state = hire_control_states[player]

    if day != state["day"]:

        if state["action_count"] > 0:
            pass_rate = (
                state["pass_count"]
                / state["action_count"]
            )

            # PASS率15%以上なら1人減らす
            if pass_rate >= 0.15:
                state["target_hands"] = max(
                    0,
                    state["target_hands"] - 1,
                )

            # PASS率5%以下なら1人増やす
            elif pass_rate <= 0.05:
                state["target_hands"] = min(
                    8,
                    state["target_hands"] + 1,
                )
        if len(unlocked_quads) >= 3:
            state["target_hands"] = max(
                7,
                state["target_hands"],
            )

        state["day"] = day
        state["pass_count"] = 0
        state["action_count"] = 0

    return state


def record_hands_actions(
    player,
    hands_actions,
):
    """現在ターンの作業員PASS数を記録する。"""

    state = hire_control_states[player]

    for hand_action in hands_actions:

        state["action_count"] += 1

        if hand_action == ["PASS"]:
            state["pass_count"] += 1

def build_market_actions(
    me,
    seeds,
    shed,
    inventories,
    day,
    step,
    melon_price,
    melon_stock,
    cow_count,
    target_hands,
    milk_price,
    milk_demand_shops_count,
    max_market_orders,
):
    """現在の市場売買・雇用・土地購入ルールから注文一覧を作る。"""

    market = []

    money = me.get("money", 0)
    current_hands = me.get("hands", [])
    unlocked_quads = me.get(
        "unlocked_quadrants",
        ["NW"],
    )

    wheat_seeds = seeds.get("WHEAT", 0)
    melon_seeds = seeds.get("MELON", 0)

    strawberry_seeds = seeds.get("STRAWBERRY", 0)

    wheat_in_shed = shed.get("WHEAT", 0)
    melon_in_shed = shed.get("MELON", 0)

    milk_in_shed = shed.get("MILK", 0)

    strawberry_in_shed = shed.get("STRAWBERRY", 0)


    # 種を購入
    melon_planted_count = 0
    strawberry_planted_count = 0

    for row in me["tiles"]:
        for tile in row:
            if(
                isinstance(tile, dict)
                and tile.get("kind") == "PLANT"
                and tile.get("crop") == "MELON"
            ):
                melon_planted_count += 1

            if(
                isinstance(tile, dict)
                and tile.get("kind") == "PLANT"
                and tile.get("crop") == "STRAWBERRY"
            ):
                strawberry_planted_count += 1

    melon_total = melon_seeds + melon_planted_count
    melon_to_buy = max(10 - melon_total, 0)

    if(
        day < 8
        and melon_to_buy > 0
        and money >= melon_to_buy * 80
    ):
        market.append(
            ["BUY_SEED", "MELON", melon_to_buy]
        )

    strawberry_total = (strawberry_seeds + strawberry_planted_count)
    strawberry_to_buy = max(8 - strawberry_total, 0,)

    if (len(unlocked_quads) >= 3
        and day < 20
        and strawberry_to_buy > 0
        and money >= strawberry_to_buy * 100
    ):

        market.append(["BUY_SEED", "STRAWBERRY", strawberry_to_buy,])

    if wheat_seeds == 0 and money >= 10:
        market.append(["BUY_SEED", "WHEAT", 6])


    # WHEAT売却
    if cow_count > 0:
        unfed_cow_count = 0

        for row in me["tiles"]:
            for tile in row:
                if(
                    isinstance(tile, dict)
                    and tile.get("animal") == "COW"
                    and not tile.get("fed_today", False)
                ):
                    unfed_cow_count += 1


        feed_worker_wheat = 0

        if len(inventories) > 0:
            feed_worker_wheat += inventories[0].get(
                "WHEAT",
                0,
            )

        if len(inventories) > 1:
            feed_worker_wheat += inventories[1].get(
                "WHEAT",
                0,
            )

        available_feed_wheat = (
            wheat_in_shed
            + feed_worker_wheat
        )

        target_feed_wheat = max(
            unfed_cow_count,
            2,
        )

        wheat_to_buy = max(
            target_feed_wheat
            - available_feed_wheat,
            0,
        )

        if wheat_to_buy > 0:
            market.append([
                "BUY_PRODUCT",
                "WHEAT",
                wheat_to_buy,
            ])

        excess_wheat = max(
            available_feed_wheat
            - target_feed_wheat,
            0,
        )

        wheat_to_sell = min(
            wheat_in_shed,
            excess_wheat,
        )

        if wheat_to_sell > 0:
            market.append([
                "SELL",
                "WHEAT",
                wheat_to_sell,
            ])

    elif wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", wheat_in_shed])

    #MILK売却
    if milk_in_shed > 0:
        if should_sell_milk(milk_price, step):
            milk_to_sell = (
                milk_in_shed
                if step >= 710
                else min(milk_in_shed, 6)
            )

            market.append(["SELL", "MILK", milk_to_sell,])

    #STRAWBERRY売却
    if strawberry_in_shed > 0:
        market.append(["SELL", "STRAWBERRY", strawberry_in_shed])

    # MELON売却
    if melon_in_shed > 0:
        if should_sell_melon(
            melon_price,
            melon_stock,
            melon_in_shed,
            day,
            step,
        ):
            market.append(["SELL", "MELON", melon_in_shed])

    # HIRE後に追加される注文枠を事前に確保
    will_buy_land = (
        len(unlocked_quads) < 3
        and money >= 5000
    )

    cow_in_shed = shed.get("COW", 0)

    will_buy_cow = (
        len(unlocked_quads) >= 1
        and cow_count < 4
        and money >= 400
    )

    reserved_market_orders = (
        int(will_buy_land) + int(will_buy_cow)
    )

    available_hire_slots = max(
        max_market_orders - len(market) - reserved_market_orders, 0,
    )

    hire_count = min(max(target_hands - len(current_hands), 0), available_hire_slots, 2)

    for _ in range(hire_count):
        market.append(["HIRE"])



    # 土地購入
    if will_buy_land:
        market.insert(0, ["BUY_LAND"],)

    #cowを飼う
    if will_buy_cow:
        market.append(["BUY_ANIMAL", "COW", 1])

    # 売却注文を最優先にする
    sell_orders = [
        order
        for order in market
        if order[0] == "SELL"

    ]

    other_orders = [
        order
        for order in market
        if order[0] != "SELL"
    ]

    market = (sell_orders + other_orders)


    return market


def agent(obs, config):

    # 状態取得

    player = obs["player"]

    me = obs["farms"][player]
    private = obs["private"]

    tiles = me["tiles"]

    fx, fy = me["farmer"]
    farmer_tile = tiles[fy][fx]

    seeds = private.get("seeds", {})
    shed = private.get("shed", {})

    day = obs.get("day", 0)
    step = obs.get("step", 0)
    hour = obs.get("hour", 0)

    current_hands = me.get("hands", [])

    milk_price = get_market_price(obs, "MILK",)

    milk_demand_shops = {
        "PIZZA_SHOP",
        "ICE_CREAM_SHOP",
        "SMOOTHIE_SHOP",
    }

    unlocked_shops = obs.get("town", {},).get("unlocked_shops", [],)

    milk_demand_shops_count = sum(
        1
        for shop in unlocked_shops
        if shop in milk_demand_shops
    )

    melon_price = get_market_price(obs, "MELON",)
    melon_stock = obs["market"]["inventory"]["MELON"]

    wheat_seeds = seeds.get("WHEAT", 0)
    melon_seeds = seeds.get("MELON", 0)
    strawberry_seeds = seeds.get("STRAWBERRY", 0)

    remaining_wheat_seeds = wheat_seeds
    remaining_melon_seeds = melon_seeds
    remaining_strawberry_seeds = strawberry_seeds

    melon_plant_allowed = day < 8
    strawberry_plant_allowed = day < 20

    wheat_in_shed = shed.get("WHEAT", 0)

    has_seeds = (wheat_seeds > 0 or melon_seeds > 0)

    inventories = private.get("inventories", [])

    famer_inventory = (
        inventories[0]
        if len(inventories) > 0
        else {}
    )
    farmer_cow = famer_inventory.get("COW", 0)

    farmer_wheat = famer_inventory.get("WHEAT", 0)
    farmer_fertilizer = famer_inventory.get("FERTILIZER", 0)

    farmer_milk = famer_inventory.get("MILK", 0)

    pasture_count = 0

    for row in tiles:
        for tile in row:
            if isinstance(tile, dict) and tile.get("kind") == "PASTURE":
                pasture_count += 1

    cow_count = 0

    for row in tiles:
        for tile in row:
            if (
                isinstance(tile, dict) and tile.get("animal") == "COW"
            ):
                cow_count += 1
    cow_count += shed.get("COW", 0)
    cow_count += farmer_cow

    hire_state = update_hire_control(
        player,
        day,
        step,
        me.get("unlocked_quadrants", ["NW"],),
    )

    target_hands = hire_state["target_hands"]

    urgent_unwatered_count = 0

    for row in tiles:
        for tile in row:
            if (
                isinstance(tile, dict)
                and tile.get("kind") == "PLANT"
                and not tile.get("watered_today", True)
                and tile.get("consecutive_unwatered", 0) >= 1
            ):
                urgent_unwatered_count += 1

    plant_allowed = (urgent_unwatered_count == 0)


    # 市場

    market = build_market_actions(
        me,
        seeds,
        shed,
        inventories,
        day,
        step,
        melon_price,
        melon_stock,
        cow_count,
        target_hands,
        milk_price,
        milk_demand_shops_count,
        config.get("maxMarketOrdersPerTurn", 10,),
    )

    # メイン農家

    farmer_action = None
    cow_in_shed = shed.get("COW", 0)

    if step >= 710 and farmer_milk > 0:
        if (fx, fy) == (4, 4):
            farmer_action = ["PLACE", "MILK", farmer_milk,]

        else:
            move_dir = step_toward(fx, fy, 4, 4, tiles)
            farmer_action = [move_dir]



    elif farmer_cow > 0:
        if (
            isinstance(farmer_tile, dict)
            and farmer_tile.get("kind") == "PASTURE"
            and not farmer_tile.get("animal")
        ):
            farmer_action = ["PLACE","COW"]

        else:
            pasture_target = find_empty_pasture(tiles, fx, fy)

            if pasture_target is not None:
                move_dir = step_toward(
                    fx,
                    fy,
                    pasture_target[0],
                    pasture_target[1],
                    tiles,
                )

                farmer_action = [move_dir]

    elif(
        cow_in_shed > 0
        and find_empty_pasture(tiles, fx, fy) is not None
        and hour == 0
    ):
        farmer_action = ["PICKUP", "COW", 1,]

    elif(
        cow_count > 0
        and farmer_wheat == 0
        and wheat_in_shed > 0
        and hour == 0
    ):
        farmer_action = ["PICKUP", "WHEAT", 1,]


    elif farmer_tile is None:
        if cow_in_shed > 0 and pasture_count < 4:
            farmer_action = ["BUILD_PASTURE",]

        elif (
            plant_allowed
            and melon_plant_allowed
            and remaining_melon_seeds > 0
        ):
            farmer_action = ["PLANT", "MELON",]
            remaining_melon_seeds -= 1

        elif (
            plant_allowed
            and strawberry_plant_allowed
            and remaining_strawberry_seeds > 0
        ):
            farmer_action = ["PLANT", "STRAWBERRY",]
            remaining_strawberry_seeds -= 1

        elif (
            plant_allowed
            and remaining_wheat_seeds > 0
        ):
            farmer_action = ["PLANT", "WHEAT",]
            remaining_wheat_seeds -= 1

    else:
        farmer_action = get_tile_action(farmer_tile, day,)

    # メイン農家の移動

    claimed_targets = set()

    if(farmer_action is None and farmer_fertilizer > 0):
        if(
            isinstance(farmer_tile, dict)
            and farmer_tile.get("kind") == "PLANT"
            and farmer_tile.get("fertilized_until_day", -1,) < day
        ):
            farmer_action = ["FERTILIZE"]

        else:
            fertilizer_target = find_fertilize_target(tiles, fx, fy, day)

            if fertilizer_target is not None:
                move_dir = step_toward(
                    fx, fy,
                    fertilizer_target[0],
                    fertilizer_target[1],
                    tiles,
                )

                farmer_action = [move_dir]

    if farmer_action is None:

        remaining_has_seeds = (
            plant_allowed
            and (
                remaining_wheat_seeds > 0
                or (melon_plant_allowed and remaining_melon_seeds > 0)
                or (strawberry_plant_allowed and remaining_strawberry_seeds > 0)
                )
        )


        target = find_target_tile(
            tiles,
            fx,
            fy,
            remaining_has_seeds,
            day,
            claimed_targets,
        )
        if target is not None:
            claimed_targets.add(target)
            move_dir = step_toward(
                fx,
                fy,
                target[0],
                target[1],
                tiles,
            )
            farmer_action = [move_dir]
        else:
            farmer_action = ["PASS"]
    else:
        claimed_targets.add(
            (fx, fy)
        )


    # 作業員

    hands_actions = []

    for hand_index, hand in enumerate(current_hands):
        hx, hy = hand

        hand_inventory = (
            inventories[hand_index + 1]
            if len(inventories) > hand_index + 1
            else {}
        )

        hand_wheat = hand_inventory.get("WHEAT", 0)

        hand_tile = tiles[hy][hx]

        if hand_index == 0:

            if(
                cow_count > 0
                and hand_wheat == 0
                and wheat_in_shed > 0
            ):
                if (hx, hy) == (4, 4):
                    hand_action = ["PICKUP", "WHEAT", 1,]

                else:
                    move_dir = step_toward(
                        hx,
                        hy,
                        4,
                        4,
                        tiles,
                    )
                    hand_action = [move_dir]

            elif(
                isinstance(hand_tile, dict)
                and hand_tile.get("animal") == "COW"
            ):
                hand_action = get_tile_action(
                    hand_tile,
                    day,
                    False,
                )

                if hand_action == ["HARVEST"] and (hx, hy) in claimed_targets:
                    hand_action = None

                if hand_action is None:
                    cow_target = find_cow_target(
                        tiles,
                        hx,
                        hy,
                        day,
                        False,
                    )

                    if cow_target is not None:
                        move_dir = step_toward(
                            hx,
                            hy,
                            cow_target[0],
                            cow_target[1],
                            tiles,
                        )
                        hand_action = [move_dir]

                    else:
                        hand_action = None

            else:
                cow_target = find_cow_target(
                    tiles,
                    hx,
                    hy,
                    day,
                    False,
                )

                if cow_target is not None:
                    move_dir = step_toward(
                        hx,
                        hy,
                        cow_target[0],
                        cow_target[1],
                        tiles,
                    )
                    hand_action = [move_dir]

                else:
                    hand_action = None

            if hand_action is None:
                cow_coords = set()

            if hand_action is None:
                if hand_tile is None:
                    if plant_allowed and melon_plant_allowed and remaining_melon_seeds > 0:
                        hand_action = ["PLANT", "MELON",]
                        remaining_melon_seeds -= 1

                    elif plant_allowed and strawberry_plant_allowed and remaining_strawberry_seeds > 0:
                        hand_action = ["PLANT", "STRAWBERRY",]
                        remaining_strawberry_seeds -= 1

                    elif plant_allowed and remaining_wheat_seeds > 0:
                        hand_action = ["PLANT", "WHEAT",]
                        remaining_wheat_seeds -= 1

                elif not(
                    isinstance(hand_tile, dict)
                    and hand_tile.get("animal")
                ):
                    hand_action = get_tile_action(hand_tile, day,)

                    if(hand_action == ["HARVEST"] and (hx, hy) in claimed_targets):
                        hand_action = None


            if hand_action is None:
                cow_coords = set()

                for y in range(len(tiles)):
                    for x in range(len(tiles[0])):
                        tile = tiles[y][x]

                        if(
                            isinstance(tile, dict)
                            and tile.get("animal") == "COW"
                        ):
                            cow_coords.add((x, y))

                remaining_has_seeds = (
                    plant_allowed
                    and(
                        remaining_wheat_seeds > 0
                        or (
                            melon_plant_allowed
                            and remaining_melon_seeds > 0
                        )

                        or (
                            strawberry_plant_allowed
                            and remaining_strawberry_seeds > 0
                        )
                    )
                )

                hand_area = get_hand_area(hand_index,)

                target = find_target_tile(
                    tiles,
                    hx,
                    hy,
                    remaining_has_seeds,
                    day,
                    claimed_targets | cow_coords,
                    hand_area,
                )


                if target is not None:
                    claimed_targets.add(target)

                    move_dir = step_toward(
                        hx,
                        hy,
                        target[0],
                        target[1],
                        tiles,
                    )

                    hand_action = [move_dir]

                else:
                    hand_action = ["PASS"]


        else:

            if hand_tile is None:
                if plant_allowed and melon_plant_allowed and remaining_melon_seeds > 0:
                    hand_action = ["PLANT", "MELON",]
                    remaining_melon_seeds -= 1

                elif plant_allowed and strawberry_plant_allowed and remaining_strawberry_seeds > 0:
                    hand_action = ["PLANT", "STRAWBERRY",]
                    remaining_strawberry_seeds -= 1

                elif plant_allowed and remaining_wheat_seeds > 0:
                    hand_action = ["PLANT", "WHEAT",]
                    remaining_wheat_seeds -= 1

                else:
                    hand_action = None

            elif(
                isinstance(hand_tile, dict)
                and hand_tile.get("animal")
            ):
                hand_action = None

            else:
                hand_action = get_tile_action(
                    hand_tile,
                    day,
                )

            if (hand_action == ["HARVEST"] and (hx, hy) in claimed_targets):
                hand_action = None

            if hand_action is None:
                cow_coords = set()

                for y in range(len(tiles)):
                    for x in range(len(tiles[0])):
                        tile = tiles[y][x]

                        if(
                            isinstance(tile, dict)
                            and tile.get("animal") == "COW"
                        ):
                            cow_coords.add((x, y))

                remaining_has_seeds = (
                    plant_allowed

                    and (
                        remaining_wheat_seeds > 0
                        or (
                            melon_plant_allowed
                            and remaining_melon_seeds > 0
                        )
                        or (
                            strawberry_plant_allowed
                            and remaining_strawberry_seeds > 0
                        )
                    )
                )

                hand_area = get_hand_area(hand_index,)

                target = find_target_tile(
                    tiles,
                    hx,
                    hy,
                    remaining_has_seeds,
                    day,
                    claimed_targets | cow_coords,
                    hand_area,
                )

                if target is not None:
                    claimed_targets.add(target)

                    move_dir = step_toward(
                        hx,
                        hy,
                        target[0],
                        target[1],
                        tiles,
                    )

                    hand_action = [move_dir]

                else:
                    hand_action = ["PASS"]


        if hand_action is not None:
            claimed_targets.add(
                (hx, hy)
            )

        hands_actions.append(
            hand_action
        )

    record_hands_actions(
        player,
        hands_actions,
    )


    # 出力

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market,
    }

Writing main.py


# テスト

In [2]:
!pip install -qq kaggle-environments

from zoneinfo import ZoneInfo
import datetime
from kaggle_environments import make
from IPython.display import HTML
import json

print(datetime.datetime.now(ZoneInfo("Asia/Tokyo")))

results = []

highest_reward = float("-inf")
lowest_reward = float("inf")

highest_test = None
lowest_test = None

highest_player = None
lowest_player = None

highest_replay = None
lowest_replay = None


for test_index in range(100):

    env = make(
        "kaggriculture",
        configuration={
            "episodeSteps": 720,
            "seed": test_index,
        },
        debug=True,
    )

    env.run(["main.py", "main.py"])

    final = env.steps[-1]

    player0_reward = final[0].reward
    player1_reward = final[1].reward

    player0_status = final[0].status
    player1_status = final[1].status

    results.append({
        "test": test_index + 1,
        "player0_reward": player0_reward,
        "player1_reward": player1_reward,
        "player0_status": player0_status,
        "player1_status": player1_status,
    })



    rewards = [
        player0_reward,
        player1_reward,
    ]

    new_highest = False
    new_lowest = False

    for player_index, reward in enumerate(rewards):

        if reward > highest_reward:
            highest_reward = reward
            highest_test = test_index + 1
            highest_player = player_index
            new_highest = True

        if reward < lowest_reward:
            lowest_reward = reward
            lowest_test = test_index + 1
            lowest_player = player_index
            new_lowest = True

    if new_highest or new_lowest:
        replay_json = env.toJSON()

        if new_highest:
            highest_replay = replay_json

        if new_lowest:
            lowest_replay = replay_json


    # 1試合終了するたびに、その試合結果を出力
    print(
        f"Test {test_index + 1:03d} | "
        f"Player 0: reward={player0_reward}, "
        f"status={player0_status} | "
        f"Player 1: reward={player1_reward}, "
        f"status={player1_status}"
    )


# 平均reward
player0_average = sum(
    result["player0_reward"]
    for result in results
) / len(results)

player1_average = sum(
    result["player1_reward"]
    for result in results
) / len(results)



with open(
    "highest_score_replay.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        highest_replay,
        f,
        ensure_ascii=False,
    )


with open(
    "lowest_score_replay.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        lowest_replay,
        f,
        ensure_ascii=False,
    )



print()
print("===== SUMMARY =====")
print(
    f"Player 0 average reward: "
    f"{player0_average:.2f}"
)
print(
    f"Player 1 average reward: "
    f"{player1_average:.2f}"
)

print()
print("===== HIGHEST SCORE =====")
print(
    f"Test {highest_test:03d} | "
    f"Player {highest_player} | "
    f"reward={highest_reward}"
)
print(
    "Saved: highest_score_replay.json"
)

print()
print("===== LOWEST SCORE =====")
print(
    f"Test {lowest_test:03d} | "
    f"Player {lowest_player} | "
    f"reward={lowest_reward}"
)
print(
    "Saved: lowest_score_replay.json"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 33.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.6/175.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 20.3 MB